# XTTS Arabic alphabet & numbers

Egyptian Arabic letter and number audio only. TTS input is **only** mapped pronunciation + period.

## 1. XTTS environment bootstrap

In [1]:
import os, subprocess, sys

os.environ.setdefault("XTTS_NB_SKIP_BOOTSTRAP", "1")

def _xtts_transformers_ok():
    try:
        from transformers import BeamSearchScorer  # noqa: F401
        return True
    except ImportError:
        return False

if os.environ.get("XTTS_NB_SKIP_BOOTSTRAP") == "1":
    if _xtts_transformers_ok():
        print("[bootstrap] skipped (XTTS_NB_SKIP_BOOTSTRAP=1, transformers OK)")
    else:
        print("[bootstrap] skipped pip (XTTS_NB_SKIP_BOOTSTRAP=1)")
        print("  transformers missing BeamSearchScorer — run: pip install transformers==4.40.2")
else:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "TTS", "torch", "torchaudio", "soundfile", "ipython", "jupyter", "nbconvert", "notebook",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
        "transformers==4.40.2", "pandas==2.2.3",
    ])
    print("XTTS bootstrap done. Set XTTS_NB_SKIP_BOOTSTRAP=1 after transformers==4.40.2 is stable.")


[bootstrap] skipped (XTTS_NB_SKIP_BOOTSTRAP=1, transformers OK)


## 2. Paths and model files

In [2]:
import gc
import time
import traceback
from pathlib import Path

BASE_DIR = Path.cwd().resolve()
XTTS_DIR = BASE_DIR / "Egyptian_Arabic_XTTS_Model"
AUDIO_ROOT = BASE_DIR / "generated_audio"
DIR_ALPHABET = AUDIO_ROOT / "xtts_alphabet"
DIR_NUMBERS = AUDIO_ROOT / "xtts_numbers"

for d in (DIR_ALPHABET, DIR_NUMBERS):
    d.mkdir(parents=True, exist_ok=True)

EXPECTED = ["config.json", "model.pth", "vocab.json", "speaker_reference.wav"]
print("cwd:", BASE_DIR)
print("\nEgyptian_Arabic_XTTS_Model files:")
for name in EXPECTED:
    p = XTTS_DIR / name
    print(f"  [{'OK' if p.is_file() else 'MISS'}] {name}")


cwd: C:\Users\sondo\Desktop\Voice Model Stuff

Egyptian_Arabic_XTTS_Model files:
  [OK] config.json
  [OK] model.pth
  [OK] vocab.json
  [OK] speaker_reference.wav


## 3. Device and dependencies

In [3]:
HAVE_TTS = False
torch = None
CUDA_AVAILABLE = False
DEVICE_PREF = "cpu"
GPU_NAME = None

try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    DEVICE_PREF = "cuda" if CUDA_AVAILABLE else "cpu"
    if CUDA_AVAILABLE:
        GPU_NAME = torch.cuda.get_device_name(0)
except ImportError:
    print("torch missing")

print("CUDA available:", CUDA_AVAILABLE)
print("GPU name:", GPU_NAME or "(n/a)")
print("Selected device:", DEVICE_PREF)

try:
    from TTS.api import TTS  # noqa: F401
    HAVE_TTS = True
    print("[deps OK] TTS (Coqui)")
except ImportError:
    print("[deps MISSING] pip install TTS transformers==4.40.2")
    traceback.print_exc()


CUDA available: False
GPU name: (n/a)
Selected device: cpu


C:\Users\sondo\AppData\Local\Programs\Python\Python311\Lib\site-packages\jieba\_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


[deps OK] TTS (Coqui)


## 4. Egyptian letter and number mappings

In [4]:
LETTER_PRON = {
    "ا": "أَلِف", "أ": "أَلِف", "ب": "بِه", "ت": "تِه", "ث": "سِه",
    "ج": "جِيم", "ح": "حَا", "خ": "خَا", "د": "دَال", "ذ": "ذَال",
    "ر": "رَا", "ز": "زَا", "س": "سِين", "ش": "شِين", "ص": "صَاد",
    "ض": "ضَاد", "ط": "طَا", "ظ": "ظَا", "ع": "عِين", "غ": "غِين",
    "ف": "فَا", "ق": "قَاف", "ك": "كَاف", "ل": "لَام", "م": "مِيم",
    "ن": "نُون", "ه": "هَا", "و": "وَاو", "ي": "يَا",
}
LETTERS_ORDER = list("اأبتثجحخدذرزسشصضطظعغفقكلمنهوي")
NUMBER_PRON = {
    0: "صِفْر", 1: "وَاحِد", 2: "اِتْنِين", 3: "تَلَاتَه", 4: "أَرْبَعَه",
    5: "خَمْسَه", 6: "سِتَّه", 7: "سَبْعَه", 8: "تَمَانْيَه", 9: "تِسْعَه", 10: "عَشَرَه",
}

def letter_tts(letter: str) -> str:
    base = LETTER_PRON.get(letter, letter)
    return base if base.endswith(".") else base + "."

def number_tts(n: int) -> str:
    base = NUMBER_PRON[n]
    return base if base.endswith(".") else base + "."


## 5. Load XTTS and helpers

In [5]:
import inspect
import wave

import numpy as np
from IPython.display import Audio as IPA, display

GENERATED = {"alphabet": [], "numbers": []}
TRIM_STATS = {"alphabet": 0, "numbers": 0}
XTTS_API = None
XTTS_DEVICE = ""
MODEL_LOADED = False
MAX_DUR_LETTER = 2.5
MAX_DUR_NUMBER = 3.0

_TRIM_BACKEND = None

def _init_trim_backend():
    global _TRIM_BACKEND
    if _TRIM_BACKEND is not None:
        return _TRIM_BACKEND
    try:
        import librosa  # noqa: F401
        _TRIM_BACKEND = "librosa"
    except ImportError:
        print("librosa not installed. Install: pip install librosa")
        try:
            import soundfile as sf  # noqa: F401
            _TRIM_BACKEND = "soundfile"
        except ImportError:
            print("soundfile not installed. Install: pip install soundfile")
            _TRIM_BACKEND = "wave"
    print(f"[trim] using {_TRIM_BACKEND}")
    return _TRIM_BACKEND

def _read_wav(path: Path):
    backend = _init_trim_backend()
    if backend == "librosa":
        import librosa
        y, sr = librosa.load(str(path), sr=None, mono=True)
        return y.astype(np.float32), int(sr)
    if backend == "soundfile":
        import soundfile as sf
        y, sr = sf.read(str(path), dtype="float32", always_2d=False)
        if y.ndim > 1:
            y = y.mean(axis=1)
        return y.astype(np.float32), int(sr)
    with wave.open(str(path), "rb") as wf:
        sr = wf.getframerate()
        n = wf.getnframes()
        raw = wf.readframes(n)
        sw = wf.getsampwidth()
        ch = wf.getnchannels()
    if sw == 2:
        y = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
    elif sw == 4:
        y = np.frombuffer(raw, dtype=np.int32).astype(np.float32) / 2147483648.0
    else:
        y = np.frombuffer(raw, dtype=np.uint8).astype(np.float32) / 128.0 - 1.0
    if ch > 1:
        y = y.reshape(-1, ch).mean(axis=1)
    return y, sr

def _write_wav(path: Path, y: np.ndarray, sr: int):
    backend = _init_trim_backend()
    y = np.clip(y, -1.0, 1.0)
    if backend == "librosa":
        import soundfile as sf
        sf.write(str(path), y, sr)
        return
    if backend == "soundfile":
        import soundfile as sf
        sf.write(str(path), y, sr)
        return
    pcm = (y * 32767.0).astype(np.int16)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(sr)
        wf.writeframes(pcm.tobytes())

def wav_duration(path: Path) -> float:
    y, sr = _read_wav(path)
    return len(y) / sr if sr else 0.0

def _simple_amplitude_trim(y: np.ndarray, sr: int, top_db: float = 25.0):
    if y.size == 0:
        return y
    frame = max(1, int(sr * 0.01))
    hop = frame
    n_frames = max(1, 1 + (len(y) - frame) // hop)
    rms = []
    for i in range(n_frames):
        chunk = y[i * hop : i * hop + frame]
        rms.append(float(np.sqrt(np.mean(chunk * chunk) + 1e-12)))
    rms = np.array(rms)
    peak = float(rms.max()) if rms.size else 0.0
    if peak <= 0:
        return y
    thresh = peak * (10 ** (-top_db / 20.0))
    voiced = np.where(rms >= thresh)[0]
    if voiced.size == 0:
        return y[: max(1, int(sr * 0.15))]
    start = int(voiced[0] * hop)
    end = min(len(y), int((voiced[-1] + 1) * hop + frame))
    pad = int(sr * 0.03)
    return y[max(0, start - pad) : min(len(y), end + pad)]

def trim_and_cap_wav(path: Path, max_duration: float, kind: str) -> tuple[float, float]:
    raw_dur = wav_duration(path)
    y, sr = _read_wav(path)
    backend = _init_trim_backend()
    if backend == "librosa":
        import librosa
        yt, _ = librosa.effects.trim(y, top_db=25)
        if yt.size == 0:
            yt = y
    else:
        yt = _simple_amplitude_trim(y, sr)
    max_samples = int(max_duration * sr)
    if yt.size > max_samples:
        yt = yt[:max_samples]
    _write_wav(path, yt, sr)
    clean_dur = len(yt) / sr
    TRIM_STATS[kind] += 1
    return raw_dur, clean_dur

def format_bytes(nb: int) -> str:
    if nb < 1024:
        return f"{nb} B"
    if nb < 1024**2:
        return f"{nb / 1024:.2f} KB"
    return f"{nb / 1024 / 1024:.2f} MB"

def unload_cuda(verbose=False):
    gc.collect()
    if torch is not None and CUDA_AVAILABLE:
        torch.cuda.empty_cache()
    if verbose:
        print("[gc+cuda-empty_cache]")

def load_xtts():
    global XTTS_API, XTTS_DEVICE, MODEL_LOADED
    if not HAVE_TTS or torch is None:
        print("Cannot load XTTS — missing TTS/torch")
        return None
    from TTS.api import TTS

    order = ["cuda", "cpu"] if DEVICE_PREF == "cuda" and CUDA_AVAILABLE else ["cpu"]
    speaker_wav = str(XTTS_DIR / "speaker_reference.wav")
    model_dir = str(XTTS_DIR)
    cfg_json = str(XTTS_DIR / "config.json")

    for dv in order:
        t0 = time.perf_counter()
        try:
            api = TTS(model_path=model_dir, config_path=cfg_json, gpu=dv.startswith("cuda"))
            XTTS_API = api
            XTTS_DEVICE = dv
            MODEL_LOADED = True
            print(f"Model loaded on {dv} in {time.perf_counter() - t0:.2f}s")
            print("speaker_reference.wav:", speaker_wav)
            return api
        except RuntimeError as rte:
            print(f"XTTS load error on {dv}:", rte)
            if "out of memory" in str(rte).lower():
                unload_cuda(False)
                continue
            break
        except Exception:
            traceback.print_exc()
            break
    MODEL_LOADED = False
    return None

def synth_xtts(text: str, out_path: Path, max_chars=5000):
    """Generate raw WAV; returns (path, inference_s) or (None, nan)."""
    global XTTS_API, XTTS_DEVICE
    if XTTS_API is None:
        return None, float("nan")
    utter = text.strip()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    t0 = time.perf_counter()
    try:
        with torch.inference_mode():
            XTTS_API.tts_to_file(
                text=utter[:max_chars],
                file_path=str(out_path),
                speaker_wav=str(XTTS_DIR / "speaker_reference.wav"),
                language="ar",
            )
        return out_path, time.perf_counter() - t0
    except RuntimeError as rte:
        if "out of memory" in str(rte).lower() and XTTS_DEVICE == "cuda":
            print("CUDA OOM during synth — reloading on CPU...")
            unload_cuda(False)
            XTTS_API = None
            XTTS_DEVICE = ""
            load_xtts()
            if XTTS_API is not None:
                return synth_xtts(text, out_path, max_chars)
        print("synth error:", rte)
        return None, float("nan")
    except Exception:
        traceback.print_exc()
        return None, float("nan")

def generate_clean_wav(
    label: str,
    tts_text: str,
    out_path: Path,
    max_duration: float,
    kind: str,
    show_playback: bool = False,
):
    print("original:", label)
    print("final text sent to XTTS:", tts_text)
    path, infer_s = synth_xtts(tts_text, out_path)
    if not path or not path.is_file():
        print("FAILED:", out_path)
        print("---")
        return None
    raw_dur, clean_dur = trim_and_cap_wav(path, max_duration, kind)
    saved = path.resolve()
    print(f"raw audio duration: {raw_dur:.3f}s")
    print(f"cleaned audio duration: {clean_dur:.3f}s")
    print(f"saved cleaned file: {saved}")
    print(f"inference: {infer_s:.3f}s | {format_bytes(path.stat().st_size)}")
    if show_playback:
        display(IPA(filename=str(path)))
    print("---")
    return str(saved)

load_xtts()


 > Using model: xtts


Model loaded on cpu in 47.42s
speaker_reference.wav: C:\Users\sondo\Desktop\Voice Model Stuff\Egyptian_Arabic_XTTS_Model\speaker_reference.wav


TTS(
  (synthesizer): Synthesizer(
    (tts_model): Xtts(
      (gpt): GPT(
        (conditioning_encoder): ConditioningEncoder(
          (init): Conv1d(80, 1024, kernel_size=(1,), stride=(1,))
          (attn): Sequential(
            (0): AttentionBlock(
              (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
              (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
              (attention): QKVAttention()
              (x_proj): Identity()
              (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (1): AttentionBlock(
              (norm): GroupNorm32(32, 1024, eps=1e-05, affine=True)
              (qkv): Conv1d(1024, 3072, kernel_size=(1,), stride=(1,))
              (attention): QKVAttention()
              (x_proj): Identity()
              (proj_out): Conv1d(1024, 1024, kernel_size=(1,), stride=(1,))
            )
            (2): AttentionBlock(
              (norm): GroupNorm32(32, 1024, eps=1e-05, affine=Tr

## 6. Alphabet tests

In [6]:
MAX_PLAYBACK = 5
seen_letters = set()
ok_alpha = 0

for letter in LETTERS_ORDER:
    if letter in seen_letters:
        continue
    seen_letters.add(letter)
    tts_text = letter_tts(letter)
    slug = f"letter_u{ord(letter):04x}"
    out_path = DIR_ALPHABET / f"{slug}.wav"
    saved = generate_clean_wav(
        label=letter,
        tts_text=tts_text,
        out_path=out_path,
        max_duration=MAX_DUR_LETTER,
        kind="alphabet",
        show_playback=(ok_alpha < MAX_PLAYBACK),
    )
    if saved:
        ok_alpha += 1
        GENERATED["alphabet"].append(saved)

print(f"Alphabet: {ok_alpha}/{len(seen_letters)} wav files")


original: ا
final text sent to XTTS: أَلِف.
 > Text splitted to sentences.
['أَلِف.']


 > Processing time: 5.132880926132202
 > Real-time factor: 3.507065704673248
[trim] using librosa


raw audio duration: 1.464s
cleaned audio duration: 0.557s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0627.wav
inference: 5.166s | 24.04 KB


---
original: أ
final text sent to XTTS: أَلِف.
 > Text splitted to sentences.
['أَلِف.']


 > Processing time: 38.165130853652954
 > Real-time factor: 2.698769611457257
raw audio duration: 14.142s
cleaned audio duration: 2.500s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0623.wav
inference: 38.206s | 107.71 KB


...

 > Processing time: 6.176636219024658
 > Real-time factor: 2.607098557226143
raw audio duration: 2.369s
cleaned audio duration: 1.672s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0645.wav
inference: 6.185s | 72.04 KB
---
original: ن
final text sent to XTTS: نُون.
 > Text splitted to sentences.
['نُون.']


 > Processing time: 22.53692364692688
 > Real-time factor: 2.804903630535637
raw audio duration: 8.035s
cleaned audio duration: 2.500s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0646.wav
inference: 22.565s | 107.71 KB
---
original: ه
final text sent to XTTS: هَا.
 > Text splitted to sentences.
['هَا.']


 > Processing time: 5.5572144985198975
 > Real-time factor: 2.765813012196726
raw audio duration: 2.009s
cleaned audio duration: 1.347s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0647.wav
inference: 5.564s | 58.04 KB
---
original: و
final text sent to XTTS: وَاو.
 > Text splitted to sentences.
['وَاو.']


 > Processing time: 16.58300757408142
 > Real-time factor: 2.783951432943228
raw audio duration: 5.957s
cleaned audio duration: 2.500s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0648.wav
inference: 16.608s | 107.71 KB
---
original: ي
final text sent to XTTS: يَا.
 > Text splitted to sentences.
['يَا.']


 > Processing time: 3.820082426071167
 > Real-time factor: 2.5298179209175045
raw audio duration: 1.510s
cleaned audio duration: 0.789s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u064a.wav
inference: 3.828s | 34.04 KB
---
Alphabet: 29/29 wav files


## 7. Number tests

In [7]:
ok_num = 0

for n in range(11):
    tts_text = number_tts(n)
    out_path = DIR_NUMBERS / f"num_{n:02d}.wav"
    saved = generate_clean_wav(
        label=str(n),
        tts_text=tts_text,
        out_path=out_path,
        max_duration=MAX_DUR_NUMBER,
        kind="numbers",
        show_playback=True,
    )
    if saved:
        ok_num += 1
        GENERATED["numbers"].append(saved)

print(f"Numbers: {ok_num}/11 wav files")


original: 0
final text sent to XTTS: صِفْر.
 > Text splitted to sentences.
['صِفْر.']


 > Processing time: 14.826872110366821
 > Real-time factor: 2.8250080363748484
raw audio duration: 5.248s
cleaned audio duration: 3.000s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_numbers\num_00.wav
inference: 14.844s | 129.24 KB


---
original: 1
final text sent to XTTS: وَاحِد.
 > Text splitted to sentences.
['وَاحِد.']


 > Processing time: 29.138880491256714
 > Real-time factor: 2.622413614380798
raw audio duration: 11.111s
cleaned audio duration: 3.000s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_numbers\num_01.wav
inference: 29.175s | 129.24 KB


---
original: 2
final text sent to XTTS: اِتْنِين.
 > Text splitted to sentences.
['اِتْنِين.']


...

---
original: 9
final text sent to XTTS: تِسْعَه.
 > Text splitted to sentences.
['تِسْعَه.']


 > Processing time: 6.976624250411987
 > Real-time factor: 2.567332522055813
raw audio duration: 2.717s
cleaned audio duration: 1.997s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_numbers\num_09.wav
inference: 7.006s | 86.04 KB


---
original: 10
final text sent to XTTS: عَشَرَه.
 > Text splitted to sentences.
['عَشَرَه.']


 > Processing time: 3.5813868045806885
 > Real-time factor: 2.371743724201231
raw audio duration: 1.510s
cleaned audio duration: 0.743s
saved cleaned file: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_numbers\num_10.wav
inference: 3.590s | 32.04 KB


---
Numbers: 11/11 wav files


## 8. Final verification

In [8]:
all_paths = GENERATED["alphabet"] + GENERATED["numbers"]
print("\n=== Generated files summary ===")
for p in all_paths:
    print(p)
print(f"\nTotal: {len(all_paths)} files")

if MODEL_LOADED:
    print("✅ XTTS model loaded")
else:
    print("❌ XTTS model loaded")

unique_letters = len(set(LETTERS_ORDER))
if len(GENERATED["alphabet"]) >= unique_letters:
    print("✅ Alphabet audio generated")
else:
    print("❌ Alphabet audio generated")

if len(GENERATED["numbers"]) >= 11:
    print("✅ Number audio generated")
else:
    print("❌ Number audio generated")

if TRIM_STATS["alphabet"] + TRIM_STATS["numbers"] >= len(all_paths) and all_paths:
    print("✅ Trailing gibberish trimmed/cut")
else:
    print("❌ Trailing gibberish trimmed/cut")

if all_paths and all(Path(p).is_file() for p in all_paths):
    print("✅ Cleaned audio saved")
else:
    print("❌ Cleaned audio saved")



=== Generated files summary ===


C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0627.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0623.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0628.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u062a.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u062b.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u062c.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u062d.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u062e.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u062f.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0630.wav
C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\xtts_alphabet\letter_u0631.wav
C:\Users\sondo\Desktop\Voice Mo